# 05 Builder (Budowniczy) | _Kamil Bartocha_ | wersja 2.0

## Rozklad jazdy

1. ❓ Problem: zlozony obiekt krok po kroku
2. 🏗️ Uczestnicy: Builder, Director, Product
3. 🔗 Fluent interface (lancuchowanie metod)
4. ✅ Builder z walidacja
5. 🐍 `@dataclass` jako uproszczony builder

## 1. 🔹 Problem: zlozony obiekt krok po kroku

Builder to wzorzec kreacyjny pozwalajacy konstruowac zlozoone obiekty
krok po kroku. Oddziela proces budowania od reprezentacji.

Problem bez Buildera: konstruktor przyjmuje wiele parametrow,
ktore sa trudne do zapamietania. Konstruktor z 10 parametrami
to "teleskopowy konstruktor" (telescoping constructor).

Problem: `Computer('i9', 32, 1000, 'RTX 4090', 'NVMe', True, 'Windows', ...)` -
trzeba pamietac kolejnosc, latwo pomylic.

Builder rozwiazuje to przez czytelne metody ustawiajace
kady parametr z osobna.

> ⚠️ Nie mylic z wzorcem Abstract Factory. Builder: jeden skomplikowany
> obiekt przez wiele krokow. Factory: jeden obiekt jednym wywolaniem.

In [ ]:
# Problem: teleskopowy konstruktor
class ComputerBad:
    def __init__(self, cpu, ram, storage, gpu=None, os='Linux',
                 wifi=True, bluetooth=True, monitors=1):
        self.cpu = cpu
        self.ram = ram
        # ...

# Uzytkownik nie wie co oznaczaja True, True, 1
bad = ComputerBad('Intel i9', 32, 1000, 'RTX 4090', 'Windows', True, True, 2)
print(bad.cpu)   # OK

# Builder: czytelne, samodokumentujace
class Computer:
    def __init__(self, cpu: str, ram: int, storage: int,
                 gpu: str = 'integrated', os: str = 'Linux',
                 wifi: bool = True, monitors: int = 1):
        self.cpu = cpu
        self.ram = ram
        self.storage = storage
        self.gpu = gpu
        self.os = os
        self.wifi = wifi
        self.monitors = monitors

    def __repr__(self) -> str:
        return (f'Computer(cpu={self.cpu}, ram={self.ram}GB, '
                f'storage={self.storage}GB, gpu={self.gpu}, os={self.os})')

---

### 🐍 Cwiczenia - problem

1. Policz ile parametrow ma konstruktor klasy `Email` jesli przyjmuje:
   sender, recipients (list), cc (list), bcc (list), subject, body,
   html_body, attachments (list), priority, read_receipt. To 10 param.
2. Napisz klase `House` z 8 parametrami (rooms, floors, garage, pool,
   garden, style, color, area). Stwórz instancje - jak czytelny jest kod?
3. *(Trudniejsze)* Ile roznych kombinacji jest mozliwych dla `House`
   jesli kazde pole boolean ma 2 wartosci i 3 string/int moze byc None?
   Oblicz i skomentuj.

In [ ]:
# Cwiczenie 1: liczba parametrow Email
params = ['sender', 'recipients', 'cc', 'bcc', 'subject', 'body',
          'html_body', 'attachments', 'priority', 'read_receipt']
print(f'Parametrow: {len(params)}')
print('Klasa z 10 param jest trudna do uzycia bez IDE!')

In [ ]:
# Cwiczenie 2: House z 8 parametrami
class House:
    def __init__(self, rooms: int, floors: int, garage: bool,
                 pool: bool, garden: bool, style: str,
                 color: str, area: float):
        self.rooms = rooms
        self.floors = floors
        self.garage = garage
        self.pool = pool
        self.garden = garden
        self.style = style
        self.color = color
        self.area = area

# Jak czytelny jest ten kod?
house = House(4, 2, True, False, True, 'modern', 'white', 120.0)
print(house.rooms, house.color)  # co oznaczal True, False, True?

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: kombinacje
bool_fields = 3    # garage, pool, garden
string_fields = 2  # style, color
int_fields = 3     # rooms, floors, area
# bool: 2^3 = 8 kombinacji
# string/int: nieskonczone, ale zakladamy 5 opcji dla kazdego
bool_combos = 2 ** bool_fields
string_combos = 5 ** string_fields
int_combos = 5 ** int_fields
total = bool_combos * string_combos * int_combos
print(f'Bool combinations: {bool_combos}')
print(f'Approx total: {total}')
print('Budowanie przez Builder jest czytelniejsze!')

## 2. 🔹 Uczestnicy: Builder, Director, Product

Wzorzec Builder ma trzech uczestnikow:

| Uczestnik | Opis |
|---|---|
| Product | Zlozony obiekt bedacy rezultatem budowania |
| Builder | Interfejs krokow budowania |
| ConcreteBuilder | Konkretna implementacja krokow, `build()` |
| Director | Opcjonalny: definiuje kolejnosc krokow |

Director nie jest wymagany. Mozna uzywac Buildera bezposrednio.
Director przydaje sie gdy mamy kilka standardowych "receptur"
(np. `make_gaming_pc`, `make_office_pc`).

In [ ]:
from abc import ABC, abstractmethod
from dataclasses import dataclass, field

@dataclass
class House:
    foundation: str = ''
    walls: str = ''
    roof: str = ''
    windows: int = 0
    doors: int = 0
    garage: bool = False
    garden: bool = False

    def __repr__(self) -> str:
        return (f'House(foundation={self.foundation}, walls={self.walls},'
                f' roof={self.roof}, windows={self.windows},'
                f' garage={self.garage})')

class HouseBuilder(ABC):
    @abstractmethod
    def set_foundation(self, material: str) -> 'HouseBuilder': ...
    @abstractmethod
    def set_walls(self, material: str) -> 'HouseBuilder': ...
    @abstractmethod
    def set_roof(self, type: str) -> 'HouseBuilder': ...
    @abstractmethod
    def set_windows(self, count: int) -> 'HouseBuilder': ...
    @abstractmethod
    def add_garage(self) -> 'HouseBuilder': ...
    @abstractmethod
    def build(self) -> House: ...

class ModernHouseBuilder(HouseBuilder):
    def __init__(self):
        self._house = House()

    def set_foundation(self, material: str) -> 'ModernHouseBuilder':
        self._house.foundation = material
        return self

    def set_walls(self, material: str) -> 'ModernHouseBuilder':
        self._house.walls = material
        return self

    def set_roof(self, type: str) -> 'ModernHouseBuilder':
        self._house.roof = type
        return self

    def set_windows(self, count: int) -> 'ModernHouseBuilder':
        self._house.windows = count
        return self

    def add_garage(self) -> 'ModernHouseBuilder':
        self._house.garage = True
        return self

    def build(self) -> House:
        result = self._house
        self._house = House()  # reset dla nastepnego budynku
        return result

class HouseDirector:
    @staticmethod
    def build_small_house(builder: HouseBuilder) -> House:
        return (builder
                .set_foundation('concrete')
                .set_walls('brick')
                .set_roof('flat')
                .set_windows(6)
                .build())

    @staticmethod
    def build_luxury_villa(builder: HouseBuilder) -> House:
        return (builder
                .set_foundation('reinforced concrete')
                .set_walls('marble')
                .set_roof('hip')
                .set_windows(20)
                .add_garage()
                .build())

builder = ModernHouseBuilder()
print(HouseDirector.build_small_house(builder))
print(HouseDirector.build_luxury_villa(builder))

---

### 🐍 Cwiczenia - Builder, Director, Product

1. Napisz `CarBuilder` budujacy `Car(make, model, year, color, doors, fuel_type)`.
   Zaimplementuj metody dla kazdego pola i `build()`.
2. Napisz `CarDirector` z metodami `make_sedan()` i `make_suv()`
   tworzacymi standardowe konfiguracje.
3. *(Trudniejsze)* Sprawdz ze Builder resetuje stan po `build()`:
   zbuduj dwa rozne auta tym samym builderem i sprawdz ze sa rozne.

In [ ]:
# Cwiczenie 1: CarBuilder
from dataclasses import dataclass

@dataclass
class Car:
    make: str = ''
    model: str = ''
    year: int = 2024
    color: str = 'black'
    doors: int = 4
    fuel_type: str = 'petrol'

    def __repr__(self) -> str:
        return f'Car({self.year} {self.make} {self.model}, {self.color}, {self.doors}d, {self.fuel_type})'

class CarBuilder:
    def __init__(self): self._car = Car()
    def make(self, make: str) -> 'CarBuilder': ...
    def model(self, model: str) -> 'CarBuilder': ...
    def year(self, year: int) -> 'CarBuilder': ...
    def color(self, color: str) -> 'CarBuilder': ...
    def doors(self, doors: int) -> 'CarBuilder': ...
    def fuel(self, fuel: str) -> 'CarBuilder': ...
    def build(self) -> Car: ...

car = (CarBuilder()
       .make('Toyota').model('Corolla').year(2023)
       .color('silver').doors(4).fuel('hybrid')
       .build())
print(car)

In [ ]:
# Cwiczenie 2: CarDirector
class CarDirector:
    @staticmethod
    def make_sedan(builder: CarBuilder) -> Car:
        # Toyota Camry 2024, white, 4 doors, petrol
        ...

    @staticmethod
    def make_suv(builder: CarBuilder) -> Car:
        # Toyota RAV4 2024, black, 5 doors, hybrid
        ...

builder = CarBuilder()
print(CarDirector.make_sedan(builder))
print(CarDirector.make_suv(builder))

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: reset stanu Buildera
builder = CarBuilder()
car1 = CarDirector.make_sedan(builder)
car2 = CarDirector.make_suv(builder)

print(f'car1: {car1}')
print(f'car2: {car2}')
print(f'Rozne obiekty: {car1 is not car2}')  # True
print(f'Rozne modele: {car1.model != car2.model}')  # True - reset dziala!

## 3. 🔹 Fluent interface (lancuchowanie metod)

Fluent interface (interfejs plynny) to styl API, gdzie metody
zwracaja obiekt (`self` lub nowy Builder), co pozwala na
lancuchowanie wywolan w jednym wyrazeniu.

Kazda metoda konfigurujaca robi:
1. Ustawia jeden atrybut
2. Zwraca `self` (lub nowy Builder)

Przyklady w bibliotece standardowej i popularnych bibliotekach:
- `pathlib.Path.parent.stem` - lancuchowanie wlasciwosci
- `str.strip().lower().split()` - lancuchowanie metod
- SQLAlchemy: `session.query(User).filter(...).order_by(...).all()`
- Pandas: `df.dropna().reset_index().rename(columns={...})`

> 💡 Fluent interface jest czytelny tylko jesli metody sa krotkie
> i maja jasne nazwy. Nie przesadzaj z lancuchami dlugimi na 20+ metod.

In [ ]:
# Fluent interface dla budowania zapytan SQL
from typing import Optional

class QueryBuilder:
    def __init__(self) -> None:
        self._table: Optional[str] = None
        self._columns: list[str] = ['*']
        self._conditions: list[str] = []
        self._order_by: Optional[str] = None
        self._limit: Optional[int] = None
        self._offset: Optional[int] = None

    def from_table(self, table: str) -> 'QueryBuilder':
        self._table = table
        return self

    def select(self, *columns: str) -> 'QueryBuilder':
        self._columns = list(columns)
        return self

    def where(self, condition: str) -> 'QueryBuilder':
        self._conditions.append(condition)
        return self

    def order_by(self, column: str) -> 'QueryBuilder':
        self._order_by = column
        return self

    def limit(self, n: int) -> 'QueryBuilder':
        self._limit = n
        return self

    def offset(self, n: int) -> 'QueryBuilder':
        self._offset = n
        return self

    def build(self) -> str:
        if not self._table:
            raise ValueError('Table name required')
        cols = ', '.join(self._columns)
        sql = f'SELECT {cols} FROM {self._table}'
        if self._conditions:
            sql += ' WHERE ' + ' AND '.join(self._conditions)
        if self._order_by:
            sql += f' ORDER BY {self._order_by}'
        if self._limit:
            sql += f' LIMIT {self._limit}'
        if self._offset:
            sql += f' OFFSET {self._offset}'
        return sql

# Czytelne lancuchowanie
query = (QueryBuilder()
    .from_table('products')
    .select('id', 'name', 'price')
    .where('price > 10')
    .where('stock > 0')
    .order_by('price')
    .limit(20)
    .offset(40)
    .build())

print(query)

---

### 🐍 Cwiczenia - fluent interface

1. Napisz `InsertBuilder` z metodami `into(table)`, `value(col, val)`,
   `build()` -> `INSERT INTO table (c1, c2) VALUES (v1, v2)`.
2. Napisz `UpdateBuilder` z `table(t)`, `set(col, val)`, `where(cond)`,
   `build()` -> `UPDATE t SET c1=v1 WHERE cond`.
3. *(Trudniejsze)* Napisz `PipelineBuilder` pozwalajacy lancuchowac
   funkcje transformujace: `add_step(func)`, `execute(data)`.
   Kazdy krok dostaje wynik poprzedniego.

In [ ]:
# Cwiczenie 1: InsertBuilder
class InsertBuilder:
    def __init__(self):
        self._table = ''
        self._cols: list[str] = []
        self._vals: list = []

    def into(self, table: str) -> 'InsertBuilder': ...
    def value(self, column: str, val) -> 'InsertBuilder': ...
    def build(self) -> str: ...

sql = (InsertBuilder()
       .into('users')
       .value('name', 'Alice')
       .value('age', 30)
       .value('active', True)
       .build())
print(sql)  # INSERT INTO users (name, age, active) VALUES (Alice, 30, True)

In [ ]:
# Cwiczenie 2: UpdateBuilder
class UpdateBuilder:
    def __init__(self):
        self._table = ''
        self._sets: list[str] = []
        self._conditions: list[str] = []

    def table(self, t: str) -> 'UpdateBuilder': ...
    def set(self, col: str, val) -> 'UpdateBuilder': ...
    def where(self, cond: str) -> 'UpdateBuilder': ...
    def build(self) -> str: ...

sql = (UpdateBuilder()
       .table('users')
       .set('email', 'new@x.com')
       .set('active', False)
       .where('id = 42')
       .build())
print(sql)  # UPDATE users SET email=new@x.com, active=False WHERE id = 42

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: PipelineBuilder
from typing import Callable

class PipelineBuilder:
    def __init__(self):
        self._steps: list[Callable] = []

    def add_step(self, func: Callable) -> 'PipelineBuilder':
        # hint: dodaj func do self._steps i zwroc self
        ...

    def execute(self, data):
        # hint: uzywaj reduce lub petli for
        ...

result = (PipelineBuilder()
          .add_step(lambda x: x.strip())
          .add_step(lambda x: x.lower())
          .add_step(lambda x: x.replace(' ', '_'))
          .add_step(lambda x: x + '_processed')
          .execute('  Hello World  '))
print(result)  # hello_world_processed

## 4. 🔹 Builder z walidacja

Builder z walidacja sprawdza spojnosc i kompletnosc obiektu
przed wywolaniem `build()`. Zapobiega tworzeniu nieprawidlowych
obiekow.

Kiedy walidowac:
- `build()` - najczesciej: sprawdz wszystkie wymagane pola
- Setter - opcjonalnie: sprawdz zakres wartosci (np. port 1-65535)

Typowe bledy do zgloszeniai:
- Brak wymaganych pol (ValueError)
- Nieprawidlowe wartosci (ValueError)
- Niespojne konfiguracje (ValueError)

> 💡 Walidacja w `build()` jest lepsza niz w setterach - mozna
> ustawic pola w dowolnej kolejnosci, a walidacja dziala
> dopiero gdy obiekt jest "gotowy".

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class ServerConfig:
    host: str
    port: int
    protocol: str
    max_connections: int
    timeout: int
    ssl: bool = False
    cert_path: Optional[str] = None

class ServerConfigBuilder:
    def __init__(self) -> None:
        self._host: Optional[str] = None
        self._port: Optional[int] = None
        self._protocol: Optional[str] = None
        self._max_connections: int = 100
        self._timeout: int = 30
        self._ssl: bool = False
        self._cert_path: Optional[str] = None

    def host(self, host: str) -> 'ServerConfigBuilder':
        self._host = host
        return self

    def port(self, port: int) -> 'ServerConfigBuilder':
        if not (1 <= port <= 65535):
            raise ValueError(f'Port must be 1-65535, got {port}')
        self._port = port
        return self

    def protocol(self, proto: str) -> 'ServerConfigBuilder':
        allowed = {'http', 'https', 'ws', 'wss'}
        if proto not in allowed:
            raise ValueError(f'Protocol must be one of {allowed}')
        self._protocol = proto
        return self

    def max_connections(self, n: int) -> 'ServerConfigBuilder':
        self._max_connections = n
        return self

    def timeout(self, secs: int) -> 'ServerConfigBuilder':
        self._timeout = secs
        return self

    def enable_ssl(self, cert_path: str) -> 'ServerConfigBuilder':
        self._ssl = True
        self._cert_path = cert_path
        return self

    def build(self) -> ServerConfig:
        errors = []
        if not self._host:
            errors.append('host is required')
        if not self._port:
            errors.append('port is required')
        if not self._protocol:
            errors.append('protocol is required')
        if self._ssl and not self._cert_path:
            errors.append('cert_path required when SSL is enabled')
        if errors:
            raise ValueError('ServerConfig errors: ' + ', '.join(errors))
        return ServerConfig(
            host=self._host,
            port=self._port,
            protocol=self._protocol,
            max_connections=self._max_connections,
            timeout=self._timeout,
            ssl=self._ssl,
            cert_path=self._cert_path,
        )

config = (ServerConfigBuilder()
    .host('api.example.com')
    .port(443)
    .protocol('https')
    .enable_ssl('/certs/api.pem')
    .max_connections(500)
    .build())
print(config)

# Blad: brak host
try:
    ServerConfigBuilder().port(8080).protocol('http').build()
except ValueError as e:
    print(f'Error: {e}')

---

### 🐍 Cwiczenia - walidacja

1. Napisz `EmailBuilder` z walidacja w `build()`: sender nie moze
   byc pusty, recipients nie moze byc pusta lista, subject wymagany.
2. Napisz `UserBuilder` z walidacja w setterach: `age(n)` rzuca
   ValueError jesli `n < 0` lub `n > 150`.
3. *(Trudniejsze)* Napisz `PasswordBuilder` zbierajacy znaki przez
   `add_char(c)` i validujacy w `build()`: min 8 znakow, co najmniej
   jedna cyfra, jedna wielka litera, jeden znak specjalny.

In [ ]:
# Cwiczenie 1: EmailBuilder z walidacja w build()
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Email:
    sender: str
    recipients: list[str]
    subject: str
    body: str
    attachments: list[str] = field(default_factory=list)

class EmailBuilder:
    def __init__(self): ...
    def sender(self, s: str) -> 'EmailBuilder': ...
    def to(self, *r: str) -> 'EmailBuilder': ...
    def subject(self, s: str) -> 'EmailBuilder': ...
    def body(self, b: str) -> 'EmailBuilder': ...
    def build(self) -> Email: ...

email = (EmailBuilder()
         .sender('a@x.com').to('b@x.com').subject('Hi').body('Hello')
         .build())
print(email.sender, '->', email.recipients)
try:
    EmailBuilder().body('text').build()  # brak sender, recipients, subject
except ValueError as e:
    print(f'Error: {e}')

In [ ]:
# Cwiczenie 2: walidacja w setterach
class UserBuilder:
    def __init__(self): self._name = ''; self._age = 0; self._email = ''
    def name(self, n: str) -> 'UserBuilder': ...
    def age(self, a: int) -> 'UserBuilder':
        if not 0 <= a <= 150:
            raise ValueError(f'Age must be 0-150, got {a}')
        ...
    def email(self, e: str) -> 'UserBuilder': ...
    def build(self) -> dict: ...

print(UserBuilder().name('Alice').age(30).email('a@x.com').build())
try:
    UserBuilder().age(200)
except ValueError as e:
    print(f'Error: {e}')

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: PasswordBuilder
# hint: import string; string.digits, string.ascii_uppercase, string.punctuation
import string

class PasswordBuilder:
    def __init__(self): self._chars: list[str] = []
    def add_char(self, c: str) -> 'PasswordBuilder': ...
    def add_word(self, word: str) -> 'PasswordBuilder':
        for c in word: self.add_char(c)
        return self
    def build(self) -> str:
        password = ''.join(self._chars)
        errors = []
        if len(password) < 8: errors.append('min 8 chars')
        if not any(c in string.digits for c in password): errors.append('digit required')
        if not any(c in string.ascii_uppercase for c in password): errors.append('uppercase required')
        if not any(c in string.punctuation for c in password): errors.append('special char required')
        if errors: raise ValueError('Password: ' + ', '.join(errors))
        return password

pwd = (PasswordBuilder()
       .add_word('Hello').add_char('1').add_char('!')
       .build())
print(f'Password: {pwd}')
try:
    PasswordBuilder().add_word('short').build()
except ValueError as e:
    print(f'Error: {e}')

## 5. 🔹 `@dataclass` jako uproszczony builder

`@dataclass` (Python 3.7+) to dekorator generujacy `__init__`,
`__repr__`, `__eq__` automatycznie na podstawie adnotacji pol.
Mozna go traktowac jako uproszczona wersje Buildera.

`field(default_factory=list)` zastepuje zmienne domyslne kolekcji.
`__post_init__` to miejsce na walidacje po inicjalizacji.
`frozen=True` tworzy niemutowalne obiekty (wlasciwosc podobna do Buildera).

| Cecha | Builder | dataclass |
|---|---|---|
| Fluent API | tak | nie (ale mozna dopic) |
| Walidacja | w `build()` | w `__post_init__` |
| Reset stanu | tak (wbudowany) | nie dotyczy |
| Zlozonosc konfiguracji | wysoka | niska/srednia |

> 💡 Dla prostych obiektow z kilkoma polami `@dataclass` wystarczy.
> Klasyczny Builder oplaca sie gdy konfiguracja jest skomplikowana
> i wieloetapowa.

In [ ]:
from dataclasses import dataclass, field
from typing import Optional

@dataclass
class Pizza:
    size: str
    crust: str = 'thin'
    sauce: str = 'tomato'
    toppings: list[str] = field(default_factory=list)
    extra_cheese: bool = False

    def __post_init__(self) -> None:
        valid_sizes = {'small', 'medium', 'large', 'xl'}
        if self.size not in valid_sizes:
            raise ValueError(f'Size must be one of {valid_sizes}')
        if not self.crust:
            raise ValueError('Crust cannot be empty')

# Prosto - jak 'builder' z wartosciami domyslnymi
p1 = Pizza(size='large', toppings=['mushrooms', 'peppers'])
p2 = Pizza(size='medium', crust='thick', sauce='pesto',
           toppings=['mozzarella'], extra_cheese=True)
print(p1)
print(p2)

try:
    Pizza(size='giant')  # walidacja w __post_init__
except ValueError as e:
    print(f'Error: {e}')

# Frozen dataclass: niemutowalny (bezpieczny dla kluczy slownikow)
@dataclass(frozen=True)
class Point:
    x: float
    y: float

p = Point(1.0, 2.0)
print(hash(p))  # mozna uzyc w zbiorze/slowniku
try:
    p.x = 5.0  # FrozenInstanceError
except Exception as e:
    print(f'Frozen: {type(e).__name__}')

---

### 🐍 Cwiczenia - dataclass

1. Napisz `@dataclass` `Address` z polami `street`, `city`, `zip_code`,
   `country = 'Poland'`. Dodaj `__post_init__` sprawdzajacy ze
   `zip_code` ma format XX-XXX (5 cyfr z kreska).
2. Napisz `@dataclass(frozen=True)` `Vector2D` z `x, y: float`.
   Dodaj metode `length()` i `normalized()`. Sprawdz ze `Vector2D`
   mozna uzyc jako klucz w slowniku.
3. *(Trudniejsze)* Napisz dataclass `Order` z polami `items: list[dict]`
   i `discount_pct: float = 0.0`. Dodaj property `total()` i
   `total_after_discount()`. Zastosuj `__post_init__` do walidacji.

In [ ]:
# Cwiczenie 1: Address z walidacja zip_code
import re
from dataclasses import dataclass

@dataclass
class Address:
    street: str
    city: str
    zip_code: str
    country: str = 'Poland'

    def __post_init__(self) -> None:
        ...

addr = Address('ul. Marszalkowska 1', 'Warszawa', '00-001')
print(addr)
try:
    Address('ul. Dluga 5', 'Krakow', '12345')  # brak kreski
except ValueError as e:
    print(f'Error: {e}')

In [ ]:
# Cwiczenie 2: frozen Vector2D
import math
from dataclasses import dataclass

@dataclass(frozen=True)
class Vector2D:
    x: float
    y: float

    def length(self) -> float: ...
    def normalized(self) -> 'Vector2D': ...

v = Vector2D(3.0, 4.0)
print(f'length: {v.length()}')         # 5.0
print(f'normalized: {v.normalized()}') # Vector2D(0.6, 0.8)
points = {v: 'origin_offset'}          # mozna jako klucz
print(points)

In [ ]:
# Cwiczenie 3 *(Trudniejsze)*: Order z property
from dataclasses import dataclass, field

@dataclass
class Order:
    items: list[dict] = field(default_factory=list)
    discount_pct: float = 0.0

    def __post_init__(self) -> None:
        if not 0.0 <= self.discount_pct <= 1.0:
            raise ValueError('discount_pct must be 0.0-1.0')

    @property
    def total(self) -> float: ...

    @property
    def total_after_discount(self) -> float: ...

items = [{'name': 'Book', 'price': 39.99}, {'name': 'Pen', 'price': 4.50}]
order = Order(items=items, discount_pct=0.1)
print(f'Total: {order.total:.2f}')                     # 44.49
print(f'After 10% discount: {order.total_after_discount:.2f}') # 40.04